# Multi-Seed Runs — CLEAN Protocol (leak-free) — bbox-only + attention

**Why this notebook.** Issue 2 of the journal-prep rebuilt the dataset leak-free
(`sequences_clean/`: crossing_point-anchored, TTE∈[30,60], 50% overlap,
N=4,906, **0% leakage verified**). The 5-D baseline is already multi-seeded
locally (**AUC 0.932 ± 0.011**, 5 seeds). This notebook multi-seeds the two
remaining variants on the **same clean data** so every row of the paper's model
table carries mean ± std from one consistent (Kaggle T4) environment:

1. `bilstm_bbox_only` — 4-D (bbox only). *Local single-seed = 0.746.*
2. `bilstm_attention` — 5-D + temporal attention. *Local single-seed = 0.936.*

`bilstm_baseline` is included but commented out in Cell 2 (already done locally);
uncomment it if you want the full table from one environment — seed 42 should
land ≈ **0.913** (the local clean number), a cross-environment sanity check.

**What changed vs the old `11_multiseed_runs.ipynb`:**
- reads the **clean** sequences (auto-discovered under `/kaggle/input`)
- **`POS_WEIGHT = 1.682`** (clean train split neg/pos = 1366/812), not 1.44
- everything else (architecture, obs_len=16, split by set, early stop on val AUC,
  threshold 0.5, test touched once) is byte-identical to the validated notebook.

---
## HOW TO RUN ON KAGGLE (read first)

1. **Make the dataset.** On your machine, the three files are in
   `journal_prep/issue2_clean_protocol/sequences_clean/`:
   `X.npy`, `y.npy`, `meta.pkl`. Zip just those three (or upload the folder).
2. On kaggle.com → **Datasets → New Dataset** → upload those files → name it
   e.g. `pie-sequences-clean` → Create.
3. **Notebook → New Notebook**, then **File → Upload Notebook** and pick this
   `.ipynb`. (Or create a notebook and paste the cells.)
4. Right panel → **Add Input** → add your `pie-sequences-clean` dataset.
5. Right panel → **Accelerator → GPU T4 x2** (any GPU is fine; runs in minutes).
6. **Run All**. Cell 2 auto-finds `X.npy` under `/kaggle/input`, so you don't
   edit any paths.
7. When done, see the **Output** tab for `multiseed_clean_summary.md` (paste-ready
   table) + CSVs. Expected: bbox-only ≈ 0.74–0.78, attention ≈ 0.92–0.94.


In [ ]:
# === Cell 1: imports + environment check ===
import json, pickle, random, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
# === Cell 2: CONFIG — edit here ===

# Seeds match the local clean-baseline multiseed run so results are comparable.
SEEDS = [42, 0, 1, 2, 3]

# Which models to run. Baseline is already multi-seeded locally (0.932 +/- 0.011)
# so it is commented out; uncomment for a full same-environment table.
RUN_MODELS = [
    # "bilstm_baseline",   # 5-D (uncomment for cross-env check; seed42 ~ 0.913)
    "bilstm_bbox_only",    # 4-D ablation (drops ego-speed); local seed42 = 0.746
    "bilstm_attention",    # 5-D + temporal attention; local seed42 = 0.936
]

# Fixed across ALL runs (locked contract). NOTE the CLEAN pos_weight.
POS_WEIGHT  = 1.682         # clean train split: 1366 neg / 812 pos  (was 1.44 on leaky data)
TRAIN_SETS  = {"set01", "set02", "set04"}
VAL_SETS    = {"set05", "set06"}
TEST_SETS   = {"set03"}
EPOCHS      = 100
BATCH_SIZE  = 32
LR          = 1e-3
WEIGHT_DECAY= 1e-5
PATIENCE    = 15
THRESHOLD   = 0.5

from pathlib import Path
OUT_ROOT = Path("/kaggle/working/runs_multiseed_clean")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


EXPECT_N = 4906   # clean sequences_clean has 4906 windows; OLD LEAKY = 1389 -> refuse it

def find_seq_dir():
    """Locate sequences_clean (X/y/meta) under /kaggle/input and REFUSE leaky data."""
    base = Path("/kaggle/input")
    hits = list(base.rglob("X.npy"))
    if not hits:
        raise FileNotFoundError(
            "X.npy not found under /kaggle/input. Attach the CLEAN dataset "
            "(X.npy,y.npy,meta.pkl from sequences_clean/) via 'Add Input'.")
    sizes = {}
    for h in hits:
        n = int(np.load(h, mmap_mode="r").shape[0]); sizes[str(h)] = n
        print(f"  found {h}  (N={n})")
    chosen = [h for h in hits if int(np.load(h, mmap_mode="r").shape[0]) == EXPECT_N]
    if not chosen:
        raise RuntimeError(
            f"No CLEAN X.npy (N={EXPECT_N}) under /kaggle/input; sizes seen = {sizes}. "
            f"You attached the OLD LEAKY data (N=1389). Upload sequences_clean/ "
            f"and attach THAT dataset (remove the old pie-bilstm input).")
    seq_dir = chosen[0].parent
    for f in ("X.npy", "y.npy", "meta.pkl"):
        if not (seq_dir / f).exists():
            raise FileNotFoundError(f"{f} missing in {seq_dir}")
    return seq_dir

SEQ_DIR = find_seq_dir()
print("CLEAN sequences dir:", SEQ_DIR)


In [ ]:
# === Cell 3: model definitions (copied verbatim from 03 / 03b / 07) ===

class BiLSTMIntentPredictor(nn.Module):
    """Baseline: input proj -> 2-layer BiLSTM -> last timestep -> head."""
    def __init__(self, input_dim=5, proj_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, proj_dim), nn.ReLU())
        self.bilstm = nn.LSTM(proj_dim, hidden_dim, num_layers,
                              dropout=dropout, bidirectional=True, batch_first=True)
        self.head = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        out, _ = self.bilstm(self.input_proj(x))
        return self.head(out[:, -1, :])


class BiLSTMIntentPredictorFlex(nn.Module):
    """Same arch, configurable input_dim. input_dim=4 = bbox-only."""
    def __init__(self, input_dim=5, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, 64)
        self.bilstm = nn.LSTM(64, hidden_size, num_layers,
                              dropout=dropout if num_layers > 1 else 0.0,
                              bidirectional=True, batch_first=True)
        self.head = nn.Linear(hidden_size * 2, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.drop(torch.relu(self.input_proj(x)))
        out, _ = self.bilstm(x)
        return self.head(self.drop(out[:, -1, :]))


class BiLSTMAttentionIntentPredictor(nn.Module):
    """Baseline backbone + additive temporal attention over all T timesteps."""
    def __init__(self, input_dim=5, hidden_size=128, num_layers=2,
                 dropout=0.3, attn_dim=64):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, 64)
        self.bilstm = nn.LSTM(64, hidden_size, num_layers,
                              dropout=dropout if num_layers > 1 else 0.0,
                              bidirectional=True, batch_first=True)
        self.attn_W = nn.Linear(hidden_size * 2, attn_dim)
        self.attn_v = nn.Linear(attn_dim, 1, bias=False)
        self.head = nn.Linear(hidden_size * 2, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, return_attn=False):
        x = self.drop(torch.relu(self.input_proj(x)))
        H, _ = self.bilstm(x)
        scores = self.attn_v(torch.tanh(self.attn_W(H)))
        weights = torch.softmax(scores, dim=1)
        context = (weights * H).sum(dim=1)
        logit = self.head(self.drop(context))
        if return_attn:
            return logit, weights.squeeze(-1)
        return logit


def build_model(name):
    if name == "bilstm_baseline":
        return BiLSTMIntentPredictor(input_dim=5)
    if name == "bilstm_bbox_only":
        return BiLSTMIntentPredictorFlex(input_dim=4)
    if name == "bilstm_attention":
        return BiLSTMAttentionIntentPredictor(input_dim=5)
    raise ValueError(name)

# Which models drop the ego-speed column (index 4)?
USE_SPEED = {"bilstm_baseline": True, "bilstm_bbox_only": False, "bilstm_attention": True}


In [ ]:
# === Cell 4: data utils (split / normalize / evaluate / seed) ===

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_raw(seq_dir):
    X = np.load(seq_dir / "X.npy").astype(np.float32)
    y = np.load(seq_dir / "y.npy").astype(np.float32)
    with open(seq_dir / "meta.pkl", "rb") as f:
        meta = pickle.load(f)
    # meta may be a list of dicts OR a DataFrame — handle both.
    if isinstance(meta, pd.DataFrame):
        set_ids = meta["set_id"].to_numpy()
    else:
        set_ids = np.array([m["set_id"] for m in meta])
    return X, y, set_ids


def split(X, y, set_ids):
    tr = np.isin(set_ids, list(TRAIN_SETS))
    va = np.isin(set_ids, list(VAL_SETS))
    te = np.isin(set_ids, list(TEST_SETS))
    return X[tr], y[tr], X[va], y[va], X[te], y[te]


def norm_stats(Xtr):
    flat = Xtr.reshape(-1, Xtr.shape[-1])
    return flat.mean(axis=0), flat.std(axis=0) + 1e-6


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    crit = nn.BCEWithLogitsLoss(reduction="sum")
    probs_all, labels_all, total, n = [], [], 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb).squeeze(-1)
        total += crit(logits, yb).item()
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
        labels_all.append(yb.cpu().numpy())
        n += yb.size(0)
    probs = np.concatenate(probs_all); labels = np.concatenate(labels_all)
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "loss": total / n,
        "acc":  accuracy_score(labels, preds),
        "f1":   f1_score(labels, preds, zero_division=0),
        "auc":  roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),
        "prec": precision_score(labels, preds, zero_division=0),
        "rec":  recall_score(labels, preds, zero_division=0),
        "probs": probs, "labels": labels, "preds": preds,
    }

# Load once; X is sliced per-model inside the training loop.
X_ALL, Y_ALL, SETIDS_ALL = load_raw(SEQ_DIR)
print("X:", X_ALL.shape, "| y pos rate:", round(float(Y_ALL.mean()), 3))

In [ ]:
# === Cell 5: train one (model, seed) -> final test metrics ===

def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=(DEVICE.type == "cuda"))


def train_one(model_name, seed, verbose=False):
    set_seed(seed)
    out_dir = OUT_ROOT / f"{model_name}_seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Slice features: bbox-only drops the ego-speed column (index 4).
    X = X_ALL if USE_SPEED[model_name] else X_ALL[:, :, :4]
    Xtr, ytr, Xva, yva, Xte, yte = split(X, Y_ALL, SETIDS_ALL)

    mean, std = norm_stats(Xtr)
    Xtr, Xva, Xte = (Xtr - mean) / std, (Xva - mean) / std, (Xte - mean) / std
    np.save(out_dir / "norm_mean.npy", mean)
    np.save(out_dir / "norm_std.npy", std)

    train_loader = make_loader(Xtr, ytr, True)
    val_loader   = make_loader(Xva, yva, False)
    test_loader  = make_loader(Xte, yte, False)

    model = build_model(model_name).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    opt  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                       factor=0.5, patience=5)

    best_auc, no_improve = -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(-1), yb)
            loss.backward(); opt.step()
        val = evaluate(model, val_loader)
        sched.step(val["auc"])
        if verbose:
            print(f"  ep {epoch:3d} vAUC {val['auc']:.3f} vF1 {val['f1']:.3f}")
        if val["auc"] > best_auc:
            best_auc, no_improve = val["auc"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch,
                        "val_metrics": {k: v for k, v in val.items()
                                        if k not in ("probs", "labels", "preds")}},
                       out_dir / "best.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    # final test on best-val checkpoint (touched once)
    ckpt = torch.load(out_dir / "best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    test = evaluate(model, test_loader)
    cm = confusion_matrix(test["labels"], test["preds"]).tolist()
    final = {
        "model": model_name, "seed": seed, "best_epoch": ckpt["epoch"],
        "test": {k: float(v) for k, v in test.items()
                 if k not in ("probs", "labels", "preds")},
        "test_confusion_matrix": cm,
    }
    with open(out_dir / "final.json", "w") as f:
        json.dump(final, f, indent=2)
    return final


In [ ]:
# === Cell 6: run all models x all seeds ===
rows = []
t_start = time.time()
for model_name in RUN_MODELS:
    for seed in SEEDS:
        t0 = time.time()
        res = train_one(model_name, seed)
        t = res["test"]
        rows.append({"model": model_name, "seed": seed,
                     "best_epoch": res["best_epoch"],
                     "auc": t["auc"], "f1": t["f1"], "acc": t["acc"],
                     "prec": t["prec"], "rec": t["rec"]})
        print(f"{model_name:18s} seed {seed:3d} | "
              f"AUC {t['auc']:.3f} F1 {t['f1']:.3f} Acc {t['acc']:.3f} "
              f"P {t['prec']:.3f} R {t['rec']:.3f} | "
              f"ep {res['best_epoch']:2d} | {time.time()-t0:.0f}s")
print(f"\nTOTAL: {time.time()-t_start:.0f}s")

results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/multiseed_clean_results.csv", index=False)
results


In [ ]:
# === Cell 7: aggregate -> mean +/- std per model ===
METRICS = ["auc", "f1", "acc", "prec", "rec"]
agg = results.groupby("model")[METRICS].agg(["mean", "std"])

summary = pd.DataFrame(index=RUN_MODELS)
for m in METRICS:
    summary[m] = [f"{agg.loc[mdl, (m, 'mean')]:.3f} +/- {agg.loc[mdl, (m, 'std')]:.3f}"
                  for mdl in RUN_MODELS]
summary.index.name = "model"
summary.to_csv("/kaggle/working/multiseed_clean_summary.csv")
print("Mean +/- std over", len(SEEDS), "seeds:", SEEDS)
summary


In [ ]:
# === Cell 8: write a markdown table for the thesis ===
lines = [f"# Multi-seed results (CLEAN protocol) (mean ± std over {len(SEEDS)} seeds: {SEEDS})", "",
         "Test set = PIE set03. Contract identical to single-seed runs "
         "(POS_WEIGHT=1.682, obs_len=16, early stop on val AUC, threshold 0.5).", "",
         "| Model | AUC | F1 | Accuracy | Precision | Recall |",
         "|---|---|---|---|---|---|"]
name_map = {"bilstm_baseline": "BiLSTM 5-D (baseline)",
            "bilstm_bbox_only": "BiLSTM 4-D (bbox-only)",
            "bilstm_attention": "BiLSTM 5-D + attention"}
for model_name in RUN_MODELS:
    sub = results[results["model"] == model_name]
    cells_md = []
    for m in METRICS:
        cells_md.append(f"{sub[m].mean():.3f} ± {sub[m].std():.3f}")
    lines.append(f"| {name_map.get(model_name, model_name)} | " + " | ".join(cells_md) + " |")
lines += ["", "Per-seed detail:", "",
          "| Model | Seed | AUC | F1 | Acc | P | R | best epoch |",
          "|---|---|---|---|---|---|---|---|"]
for _, r in results.iterrows():
    lines.append(f"| {r['model']} | {int(r['seed'])} | {r['auc']:.3f} | "
                 f"{r['f1']:.3f} | {r['acc']:.3f} | {r['prec']:.3f} | "
                 f"{r['rec']:.3f} | {int(r['best_epoch'])} |")
md_text = "\n".join(lines) + "\n"
with open("/kaggle/working/multiseed_clean_summary.md", "w") as f:
    f.write(md_text)
print(md_text)


## After it finishes

1. **Output** tab (right panel) →
   - `multiseed_clean_summary.md` — paste-ready mean ± std table.
   - `multiseed_clean_summary.csv`, `multiseed_clean_results.csv` — same as data.
   - `runs_multiseed_clean/<model>_seed<N>/` — each run's `best.pt`, `final.json`,
     norm stats.
2. **Sanity checks (clean protocol):**
   - `bilstm_bbox_only` should sit around **0.74–0.78** AUC — confirming the big
     drop from the leaky 0.889 once the static-geometry shortcut is gone.
   - `bilstm_attention` around **0.92–0.94**.
   - If you uncommented `bilstm_baseline`, seed 42 ≈ **0.913** (local clean number).
3. Download the output and drop the files into
   `journal_prep/issue2_clean_protocol/` next to `05_variant_comparison.md`,
   then we lock the final model-comparison table (Issue 3).

**Expected runtime:** ~1 min/run on T4 → 2 models × 5 seeds ≈ **10 min**
(3 models ≈ 15 min if baseline is included).
